# Phase 3 — defect classification (champion vs challenger)

Severstal steel defect data. Two questions: does this strip have a defect at all, and which
of the four defect classes are present.

What phases 1 and 2 measured, and what it forces here:

- 6.41% of defect images carry more than one class (427 of 6,666). A 4-way softmax would
  have to pick one label for those images, so the head is **4 sigmoids + BCE**, not softmax.
- Binary has-defect is 6,666 vs 5,902, i.e. 53/47. That is balanced, so the binary baseline
  gets no reweighting.
- The real skew is **between defect classes**: class 3 has 5,150 images against class 2's
  247, about 21:1. That is where the imbalance work belongs.
- Images are 1600x256 strips. A square crop would throw away most of the frame, so we resize
  to 256x800 and keep the aspect ratio.

Champion is ResNet-50, challenger is EfficientNet-B2, both ImageNet pretrained. Same input
size, same split, same augmentation, same schedule. Only the architecture differs.

## 1. Setup

In [4]:
import os, sys, io, json, time, gzip, base64, hashlib, random, zipfile, math
import numpy as np, pandas as pd, requests
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    os.system('pip install -q albumentations')
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

import cv2
cv2.setNumThreads(0)   # workers do their own threading; cv2's pool fights the DataLoader

WORK   = '/content/steel'
OUT    = '/content/outputs'
for d in (WORK, OUT + '/metrics', OUT + '/figs', OUT + '/ckpt'):
    os.makedirs(d, exist_ok=True)
os.chdir(WORK)

SEED     = 42
IMG_H, IMG_W = 256, 800      # half of 1600x256, aspect ratio preserved
N_CLASSES    = 4
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('albumentations', A.__version__, '| device', DEV)
if DEV == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          '| %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU.')
print('cpu count:', os.cpu_count())


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

torch 2.11.0+cu128 | torchvision 0.26.0+cu128
albumentations 2.0.8 | device cuda
gpu: Tesla T4 | 15.6 GB
cpu count: 2


## 2. Data

The Severstal images are loaded onto the runtime once and read from local disk thereafter.
Integrity is checked on contents rather than on an archive checksum: the cell asserts the two
counts phase 1 independently measured, 12,568 images and 7,095 label rows. macOS AppleDouble
`._` sidecar files are removed first, since they end in `.jpg` and would otherwise be counted
as images and then fail to decode.

In [5]:
from google.colab import userdata
import tarfile, glob

DATASET = 'yybyhb/severstal-steel-cv-final'
BUNDLE  = '/content/kaggle_bundle.zip'
HDR = {'Authorization': 'Bearer ' + userdata.get('KAGGLE_API_TOKEN')}
TRAIN_IMG = f'{WORK}/train_images'

# A partial archive from an abandoned browser upload would only waste disk.
for stale in ('/content/steel_data.tar',):
    if os.path.exists(stale):
        os.remove(stale); print('removed stale', stale)

if not (os.path.exists(f'{WORK}/train.csv') and os.path.isdir(TRAIN_IMG)):
    if not os.path.exists(BUNDLE):
        t0, total = time.time(), 0
        url = f'https://www.kaggle.com/api/v1/datasets/download/{DATASET}'
        with requests.get(url, headers=HDR, stream=True, timeout=3600) as r:
            r.raise_for_status()
            with open(BUNDLE, 'wb') as fh:
                for chunk in r.iter_content(1 << 22):
                    fh.write(chunk); total += len(chunk)
        dt = time.time() - t0
        print('pulled %.2f GB in %.0fs = %.1f MB/s' % (total / 1e9, dt, total / 1e6 / dt))
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(WORK)
    # The dataset may wrap the archive rather than the files themselves.
    for t in glob.glob(f'{WORK}/*.tar'):
        with tarfile.open(t) as tf:
            tf.extractall(WORK)
        os.remove(t)
    print('extracted')

# macOS stores extended attributes in AppleDouble side files named ._<name>. They end in
# .jpg, so leaving them in place would feed the loader thousands of undecodable phantoms.
junk = glob.glob(f'{WORK}/**/._*', recursive=True) + glob.glob(f'{WORK}/**/__MACOSX', recursive=True)
for p in junk:
    try:
        os.remove(p)
    except IsADirectoryError:
        import shutil; shutil.rmtree(p)
print('removed %d AppleDouble/__MACOSX entries' % len(junk))

files = sorted(f for f in os.listdir(TRAIN_IMG)
               if f.endswith('.jpg') and not f.startswith('._'))
print('train_images:', len(files))
assert len(files) == 12568, f'expected 12568 images, found {len(files)}'

rows = len(pd.read_csv(f'{WORK}/train.csv'))
print('train.csv rows:', rows)
assert rows == 7095, f'expected 7095 label rows, found {rows}'

# Geometry is measured off real files, never assumed: 409600 pixels also factors as 128x3200
# and 640x640, and the RLE codec is column-major, so a transposed shape raises nothing.
from PIL import Image
sizes = {Image.open(f'{TRAIN_IMG}/{f}').size for f in random.Random(0).sample(files, 20)}
assert sizes == {(1600, 256)}, sizes
print('geometry confirmed (W x H):', sizes.pop())
print('data ready: OK')

pulled 1.26 GB in 11s = 111.6 MB/s
extracted
removed 12570 AppleDouble/__MACOSX entries
train_images: 12568
train.csv rows: 7095
geometry confirmed (W x H): (1600, 256)
data ready: OK


### 3. Per-image labels

Kaggle's `train.csv` has one row per defect instance, so a two-class image appears twice and
a defect-free image does not appear at all. Classification needs one row per image including
the clean ones, otherwise the class balance is measured on the wrong population.

In [6]:
def defect_area_px(rle):
    if rle is None or (isinstance(rle, float) and np.isnan(rle)) or not str(rle).strip():
        return 0
    return int(sum(int(x) for x in str(rle).split()[1::2]))

raw = pd.read_csv(f'{WORK}/train.csv')
raw['area'] = raw['EncodedPixels'].map(defect_area_px)

rows = []
for iid, g in raw.groupby('ImageId', sort=True):
    per = {int(t.ClassId): int(t.area) for t in g.itertuples()}
    rows.append(dict(image_id=iid, has_defect=1,
                     n_defect_classes=len(per),
                     primary_class=max(per, key=per.get),
                     defect_area_px=int(g['area'].sum()),
                     **{f'has_class_{c}': int(c in per) for c in range(1, 5)}))
index = pd.DataFrame(rows)
clean = sorted(set(files) - set(index.image_id))
if clean:
    index = pd.concat([index, pd.DataFrame(dict(
        image_id=clean, has_defect=0, n_defect_classes=0, primary_class=0, defect_area_px=0,
        **{f'has_class_{c}': 0 for c in range(1, 5)}))], ignore_index=True)
index = index.sort_values('image_id').reset_index(drop=True)

n_def = int(index.has_defect.sum())
print(f'index: {len(index)} rows | defect {n_def} | clean {len(index) - n_def}')
for c in range(1, 5):
    print(f'  class {c}: {int(index[f"has_class_{c}"].sum()):5d}')
multi = int((index.n_defect_classes > 1).sum())
print(f'multi-class images: {multi}/{n_def} = {100 * multi / n_def:.2f}%'
      f'  -> head must be multi-label')
assert len(index) == 12568 and n_def == 6666, 'index does not match phase 1'
print('matches phase 1 counts: OK')

index: 12568 rows | defect 6666 | clean 5902
  class 1:   897
  class 2:   247
  class 3:  5150
  class 4:   801
multi-class images: 427/6666 = 6.41%  -> head must be multi-label
matches phase 1 counts: OK


### 4. The frozen split

`splits/train.csv` and `splits/val.csv` were generated once in phase 0 and committed. Every
model in every phase reads them, otherwise the champion and the challenger are scored on
different data and the comparison proves nothing.

They are embedded here as gzip+base64 rather than regenerated, because regenerating depends
on the exact `sklearn` version's `train_test_split` RNG. If that ever drifts the two models
silently end up on different validation sets, which is the one failure this phase cannot
afford. The cell also regenerates the split and reports whether it still matches, which is
the reproducibility check.

In [7]:
SPLIT_TRAIN_B64 = '...'

SPLIT_VAL_B64 = 'H4sIAHkydWoC/12dWa4tPW6l32ssRkF9M5qCGsrIgg0YsOHx1+JOoM4Xzv++nJWxFQqJIhcpivrHv69/tf/zj/sv//kf//aP//pfIYR0zsz7f//f//jXf/nv9W9C4hxph0Hkxp1r/SD5hpmB1DvaSfWDrJzPAzJmSb0Qec26evCHxHTTvK8AqSWctSOQ1lcvwYCM2GZ9E8gOJdcQ/pDUQzwp45k0bJdy0XKa5aV8+Ku1Wiqdz+x0dj35g9y8FvqcY+zzNCPyTqkb45NLi28Y3pXb6TcOjEYep85U8a4841WfG5Ha+y34rrx2sWwLyF19vgukhFrfzehhSXrTWBdIneWGwmd6GzHVBGTM1crtQFbpvRv6U9S7MytarnFYn3ymxhfuoPzUXF7chUjJ0SQfRG5sNbDl8tatEeNTZ079WSYyY3gVo1H3Sb11tnxCtVX4q+tiZ++D7HMeVkqLJ5cQ0XLLb8WWMcut7hzTQztt1NFjTEROO2VhfNqS9PZeiUgKyzAid+S9ID/Naj2bo9FeqXoI8txjLvtEIrW3VrkKJDxP34Z39VVb/R/I6S0FvL1vffraGI1u453VM5G7pUkqEVOfN0ZjSKLPixix0XyCDFprjDB2XPzVWGYjoYfjjiDZQH/Um3EPJXM8m68VrOUZ6rNA7TdjCVcrCkiKpT5++0wtlvWMyBs3N7aT7cTTID+z9HF7wHzNenqqH6SlvuN+RIqEqkQiUnaJUjelZoNWD5BRTh6B7Yw2Z3v81Qzz1Mw+z2k1L/5qp+Y6EsiLmh/Dt69crOf7iPQXVppE9taKw1wsWYsVGr59tZxKSpAfNwTPOt6+Rg5tc+SlmvOc1M9S6u1ervd1zI56BMTCkFZAOzvOmRc1rVZy7fWgnV26TwWfKWeY1CaQKoOVuSqF6Ecdb99jvjmpV/eyeHtkf/aNKX6e2XbyXBifffIbxyBj+665PmtnmxW7H+SdJ2WHPp8UT9kFfT5l1h0PWj7t1Vo/v9p1hkcddU4pYXANnpvSls4GYs1epNXTI8UG7c7NSb/JfCaPMSa1hHOUux9/VUZrlrFOb5NeW1yDtw9pMmr+262mNfFdV+K082Q7S508sROJJxSynbu7ncL1ZVorL2/IvMXQ1Ev02eoUu+mQKGvv9TIgz9IjEnpqUQlzjjnymdvutMu3W5Jm75tIqdkC3v5kLUvjqnz5WrkXc/p2baclPqM/RyRzeCabF6CjYgjXcgMHiCGW2BKYVRT5aDFhfWl0Vj2vBSLv9gXeEmUe4mzGd80ucbl81xI13bBfMewkzrSJOAcY0L1CtHTrYn/EbbQsKpEpvfFp58436he5PYcP8qIYyKc/701/6A+JRWKZ2yRiYxpHIzbRlAy2Iyba65RSBDJfkigCSUG0ZPQHJOc0A7ixkF7V7UVkiO1wBpOYTiyTz/RVloYfyOhH/xUgWrpLtPYPyanGuAe+XWz5jM2RzzLLKUbMe+671tKITH1nm/zVmTLWl4gImmwufiV/QwoI6z2WUkVdxgCiuZr786suyvqZ5TLymvnzjKjXqYnIkaHJ5QAxUbQG9hU1Fnd2sHcheZew+IwMrGQV31WbbCOtVZQY9Ps221k75GOQXrkoWpOF7ViNaYEnCDkmzYY5bSHnfvIH6UvqB79q4khjHD6Te5RRK0SOTG7Fd7Uq9RfCB5lhGHvY6qr1Voxha0njcyIReSE78u3Som8mIn2JLzdIXZt7S16wUjQxkrzPu056axm/VAZsHs5Fl0SJs3UiL6cEVhB7EmkK0LRCdk8ZdjBqQp/8OGikXusezTCGXQOfX+W7RNF643xpRM8NOxAR9TRwLSFzn0WdOWQEg4FZicWE7K4tESkFG/iKUaT9Jmy3ENtzDLbTwn3yHojI/7OHcR49t10HZHXsVGSt0OcZJdLyKIBowa808BWzLolP5zPt5TzBEzwCUtOGPRXzu2vIYHyQMybsYBSt2rbBbYS4HulsWcQqXeqW+UxrgNp4pZDdkySy5V5RfrRwu+RlE4l5jYSW1ym5G3hvXFcmjMxTiNzcze/SBzg53URkYfdGf2TxSnrgbHFXkfWdiTRZq4cYiOS/tWkb/dkzyGDRymw5RdIlGEO9WL3hV+x7TpOcEXkycQfftcVi5IJ9EEniBf8RIjfgfN4l+lwe7dd+vbrf+IecKH548weZWhX3EjlL78d3nbTyHBP9OXnVscMXkQJ6bKfI4TCOz1myTfFhLs6WoT4cDXHjEET1iJR1U8Ms3yJu8+B/CVk1DvgXQnaIkRr7VslLJYuT+Ggxb76rPvm1m/1ps2z52UDkRU7GN+IVrZP5fETeqW3wV/KVV1/szxny7cia7k1addTP98oSDvin0YLEPtMKWxpvMaYnRJLQqW2cGovYdiJLE4YYWrR2w/jwTOt7thqIzFpOQ3xVSMu3P0iCFq0UYrAP4r4nxse2yNhe7OGR43Yj3yUTlwp5gv6UZJBvvLqe1A3m9PUtaxE+iM2sTgIZWjydNu7NUuQBYQzfvpY6v/QdEWwj33hH/p9RVt+R9LTCdixfqUT2x64IqvG7ZE5jgebXEtBPcu9A0nCiWYBk+b0ZWiuFss87YHopDBkDa2xnNhEr+HEprDHLWWxnSxDmZH9uL4mRQJHX/saGN5qiXOE5D5+pUiwVs5Nk8p8s8yNiHgwzIDI7KcM3T1EOtQxoIjJmqo1vf3P2kD+/2uk+WPOUxPeNspH0vxUyrEOSX7Dl7aE/qfZjsTcgXYx6It6iL78lMR6eslvzm/FMTrYrNa1e5TsICy3nu6u+Av3JVtraL36QpdXMlsVy5T5gfEqUZM5BxCPSGawglSLDd2FlUqlpyAsJRGbv1tFn0eByGD3QmvRwSvggu8vIoM/FR3mBr6YiM3Rf5jNXX5oRkU7lldcHdEKqQXNBD/Hna6X+8F01WbwG/ytJ6w8ZUHxX3WKiAZFAIbJgbbJlE2PtA+PcYhApgDUX4nQC2i+1lN7o8OOE9CQLim8XV1/hcpbbKEWmkc/soPdTDpt4Z0jwl2UT5SilwXZuTDvCd5AKO0fu1yDySghgp+pfsrQphz2JbciCAZHo6ncY1S6ybtH4ri69bpnPbDkmAd5f6ndWV0FALM3YBr69P2njEdnO88DoDESWXF/Dr4b8WqlajLwMp5g37I4Qid2kPA/pwmWGLx1ig6HB4qchTlky4mxCloxIhSQMmWAZDPZn3T0WIpPJOf99ie/ao17uCgk5W4KIeR8yKZcRRVEmCyF1IlGf2qixhYhXdvR5ppIWuVaaJbvnhrWjuZExxw5LcvOuxYovlU8py5wfEXk3pcUP0lOgJplHXu2FrUzz1eDsnIjUdcP+qZSz/h7gJOnH8yalZdXbZy/49tWlyQLfteQQtgUuKm9QrGCxnR1ClZDj7fIm7tzLiDx9Fkd1F2m7S4u2pSJHwW6XEPlJkVK3ax6FHEA+kofjEyRhDykfA19N+3SPAOFLj3S/6Dtk9ZQigw92KmTHEMkuTpH7wzhAOrLLb4GzCbHUHlfTGc4PaU/Psffx9bS6avpyiSNfphm/9AZ31sEqk4xQEGV9H6TKG8fsyAOQf3ozkbc8HAckV60Dcpub2wl6IZA62uPeaLquOKx+EFGXBS8yXRHheT/IiWUU7BMJEWUrtMsSiyeyxWfkw0rjsB15SLNx5K8I4mOcP1m49V7sfyXrMucP/leSfTvypCC99u6WX4n5ctnNht3b9Jp07b4f5Be3MSLyoC9Xyus5Fy1MIKKm7VAS3jSxrQBJ0KhL+WFV5hBkG6RxgGhdlgGml0XpZLBgC3IoYeSDfSIhctMyYoNZa7S2+Yio3ZECW55d3kzk25dvy3ze7m5tAG/JYsLV4mR/pDPsflo2af6OGcy+Yx8eNEmO4cij5rtidAuLaJiQ19ynB5K0mA0rV4h4b0Z8TMjZJ4FHSSzl6WmpACk537z5rioGFOFT59hKLG+wh2LYh3su8tWrlBZWU3aXo9PPzSIl4lLwhXNab+0G+clSiPUdsFx5uFIj8ov+kJyGVBnyZLKH60R/iBQ53RXMM8s7lG4FQxNS5XVDG6sZjcVIGLESJD5141elr6amIFHS+2sNaC2tCRmQgai1kC1ezlEt5k50ZMsmrpVhBzXjsmkB3k2uQQpgInKSZeBannMQ6e4WfRARRMt4l5a/dA3ivbmWLgRxACHLTp3ocxURngl+XK5DWrVkvuuMsLn/LmSuvMEzs7R8lUcGqWuxPg+xAEnrG1fPreiZMtDn1m7uh2undVeAm+/qfdzyaacPH3jMYNPspRL49qEvrWDLuUlLiD2wHX2W70YAWXlagScuOjRajYvv2ne89PmVaVC5d5N7CPN07C6JREkZPsRpc281yAHLRK4kLPIZD3iexGemVuEBp81952yXWlQ/Wpt7ZLlfDXuo7KEUlKxVIiK/5MG+i9g80fWCMRTHFU2iJhnNt2K5ckebQ3YGsir689qFt5XHalnu6SYic2aTLR/f30VML0/3+pmt4WloIQZkZ2UNzyrl8JmcpX8ORmyWI46GGIi86TziQqQiTyfrPbIdCfOK8NHy1DJJmV86txacwebmqU+QwwNpmVrH9cK3ysv3cgJ2B4T8gp6YwZVSOmsQaRKyxblwvSbPjc/Ir5LyQ3+WpSe+WonMexq/Yr0ZyqbeWG8PmW6M4U57yvLhu3aRs9nBSbIH3j47hlnEr0mEoX/22+KnESOvAcyyDvjV6fPKr+Uz7j9PWoczd9/cUxDiDBrehKiGfOMeP0gTe7YPItMYaQevfGXx8EckrhDA0LLEvWTG5/OdKx5xRCDH0z7Y5/veyYHPmEhK+mh+y2JkAd5oln4Kp8FH86DRtMbVbcPmOtiJyHb0WRPMM7sOG41c4snKhM319f6ZUgZpeVfa94CvCjm9dWSCCbHQBjX2E9OxynXxnvRxhBYtIa9wNjLliqd1ymJMIDXuueoHqaHLWwBy5BYc5FU6+QnivvhV7LH1CNkocW9PP7pAzih1Lf7qyhGPrxNxgjjZjnU5yPB8S3xBKwExvZJCkZVDTpeQOeVW4ivk/di64FpaXWfsiAycksTP5L0sIrYtIT+qpJmrbAieEa0U44E3WkTaRpReAKK5epnfleWbrwM9X2TgTcaRLbcirwBsueQjixqwloumx5wqACnJiTlGo4hb7HSItFPsIW+5lKWRvtipETLlDGM1CblTn0FELtGr0FrlF6u80PylWBseuiUyizGrvMjttvUQVStiolJAF+2IfYU2OfJVBnZWjnNt8pqYa1TE0MI6CX2uc5SQkAvqyBvb+IzWxGH0oNS9YgiUw3qk7dLnVyYfNsMXFrKfR7uAvDT1O/Swxez+xfkgUrWIcP5yyqXIMc6tzixHG6PRxkw7IwqqiZBsiOgC8W3gUtiyfIvZD99+Ne0b3lZpFrTgJ5/RWkrMKi9dc2MB8Q0ha0qA8aUyZ2LQlNXus/rRElL8pkEcRFaQ+4UvlRcVR4P1lAsizzJtjHN/8swnv2voVfciMimky5mA/y7EbuMOr7vc/bu6PW4aJ/s8dqieWUnkyTlHfkuRjq+dNkVIac68/xC9poiBYXamRFxs54MMPYFYSplXfGcg3qJ1YvNk5FqXJUopE4ovXeVESQJmZ9WU5bxg5Jeon5QNvmL9+Fhiy2/a3IgRyQ0XoZ7sz443LUbjyxZ734/aT9wwadCIXHV6wa8sevmQK/CI/FK9gZwgynHh9es7Q1wFHEmI77Aguiu3QYs7YJdTSJXdpS042XOUFt8l7hCZOSMe3D1ZhC0PLd3W+SsRtJrhKQiZ48OEi7xwLfnAPr+8RaIHkTXjoITfMIamHd+uD1ipVvxKrsWVHkXLVyr8NGRQlDvV4bs/yN69UrfI/diyO1hN+sH1tCkiNddG2dD608uoE+4rs330mMkHSTzH4UiSs4fVbcU9es6FleOnWtBDq7KVF1528fytOSgt1ncVb+Kv+gvGEyvFxq4rIDNNiHXJM1teMg2dttJk3FfnqNo5w9bnu67okC22Yzmdwvl6TVO2sV8gU1nkW0Hb1OAZoxkxtKpFUB5tpVzjpMUTPohG8Rz+qqZRGO8VFSxpR9hKd6tOMuixGtzscA9IfmjV2y6f0bvfwXxVT95dC55mjSnl1sCxq6iyKDW+Xcj0seczO4X4IM81nu15Z3i7lrtV6vDqW/KxwfsT8iSL4MY1+f5Sw1quSdISHqIQGi+LqXb+aroNhn9a08q1R6xcISvOheiTFsWyPXEapWb3vAc0QM2ScAtgVn6UauWFHSgttzAWs1AcsdixKoV49iH2C4RoCTR4AY5IIcPC1ty7dPTmu/oceowtd9MEYhevZo8EGjxWR+yT91Xzjp6PaR/EPD0dyAmyghzV7HsK67DPd8vMPCJmdjLWoFT6dC6DduQU7RfguQiZMinQAFXujXwiyoYM51xGmS9dxnOCpchTlkY80DZVBuMYI4HVFb1cNMyyVMCWMSfyZKmZ6fRDXoPPWNXQvg0+o5B3U0I2VNVcnVMQq5Sn7uF3ru56RzdGuoTI0ztgsL4T+HyIgLzVX0bkpDqn3Z2ru2mS5XZHIltcD/vUVcyryysh0or8edjlKrqm76COalLpYWDnUXrfY0LUP0269+3JZ94W7+eodj/A8rh2ZIP1nYlIdx/oVCJ3f87D1j72agV2WfphSSshTlJlPqydyl8t+c8T1qH206VNqAG6zKdWM/vzqtlAjKiOsMLtiPs5IncGXF0MUpOTwbXqEKv9MM86zLeTOe+zPzm1YPjVl4C4KJ/Zv+Nej4gcDGaPyLEKYd3IZ56W/KVuEYOVx8U5FV9dc8PCVjlfQQwNfZa/YZ8d5/o7zrNoB8UNx56Z73ohCYbUSYOPemDfhcwYGA2rojayBvDE1d3mTmQmYmIJHMM9hlzoyF/NlCp3OT26MA5zFIXscWxBErQC7jBqkn3jLg/ReCEiog+53/UsOZ4L+/hyYWXhG9fyMc8aRB6RPJt4BqNPjlhlpncV85K6gw/rSDub0nujxC6xPzd67J0y7/mboWP/q3qOq0QIb7+ltHRpC25bqTHao8FJpq/F+NxZnSbwV/IPc5h811taT2QOnpMrtoV2TPw5XOofj9CIGwN5Ict4Iue/vigXpCU+E7vcfuQj+XbP1legPy950h3n9EmeVuM6fUuewViQsWd7hYRc0Pqe/GdD7koLnq6VIL1CmutwIvLHPpmoTfZNPhv2SlpYZ4WMKGiLQa7W4K9iOuFORI2EeFUB5KE1P66zJjiJuwTDNxKBjOgHA8YH2WFNvn2LpNDKCNEc1sh27i9W+jfO0iN+bA06SoZoiEgdIlLyk/mrLZX2+gHrFuJnbLCjKuSlxwyBluqRlYl8e8ue+Yo+p7HWS5fPSP2FCA3Z0l5iMpXvEtWR+cI4+9HkMGHxW05BPjQ4dsvFt5zA9ITUrSXHX/khn9L5q9ZD2JPPnBTnwcoVsiSXgb/Sqn0Vu+Sy7knyC40tpJ1o7YOM2pkN3op8m9ORn9DkMsU6EbtopYscT/iwQkYSI4OElxPbiuDGrbwqH5vfXt5+dSDi0WpYfkYMEl5jlL2AXZbPK/LZERtsIiVPMg1JqDPkGcAYW5W+PLXymXNPq4ktm0SlRfbHkw87uERrWpKTsZTW4jt7ICYsYnWk16B7W/PN7AYvqbVRppYpxlnu0OyTc9HM8+kT32UyDouz3NW/ssGWWxdRTwG+lRC5bAW5sq3LwtbJMez6+LKxM9tkgscpyJORAi1dpAxS119bY/DtI8WYO1ickHb6hl4VIkYi7gBk92/FkjbOrIlVO4ScUJiLLtu6QmQuVtOcW+QJiKYlul+HJyVElrtTQ8rJ9joUbEcERDoaXzGlHmU/MRcyelmewCMiQ8QzqkLMvLQIkLbj5Sl131oqcyKeKaSkfvsHkSIrOCEiYj6nnFT2cGmk9/0gcooa5WfeWrRYMMvTPCHzcnxMtI3xMSEyB2/w7XJk/EAckCc+vaj5l+x/MVhqMQmnwvCA2spq+CGntC0/wsa4TVvVLTeYcPNjZZdVaOSUvFc2JXxJomwc/GrLMM8Df1nIDVmMFUhKSwQNsyN2ao81Q4ScqxnE23fJYVxEGJq8r9kPMl48NWPNSju4Rbkj89XddRdT4Azue3u5k+3cJ3PB9bXNZb6xz3Zif+BaQu5eM/Dtb8pBBIsTsqfH9f4QmUXpNvB5IW33i9MWMihdmhQRPC9pFEKl9B4JmFxbzIWk8sjdQn+OKNueyF9tZ165log2txuLFi7H+Yrzurol4od+kLfTbr5l8wx48523/LHm1xPxB/nGlZ3pCzs17UrmpAHZ8jnrMSbj2zsjGq2DBc9q54iZ/93nB5FTvXHmsVmWk9Zw9k1IyaJaREqJJxvfJRvoG2dA9JPLM2LNpmxRrvzV9pxSxG2EbEkMcvmE/AgQe7j1VCCr9L2uymyx9tLxQ2uYr1c17Jfa77l8G6XlLRmHTrvsnuityGvqzp+b4e095K2li320Hko9cqkrkDZMxjsAGfIuGAXtYcUbmSPUw91xZUSxhMgdY66jEDM3s39IjEE2A9FmIb3K0S5A8hufU/M91poX94l6dKLQPr/Se/bDuujSockLIgGRcZqMxclnysurJhEpfqIA4yO65tUQ+MyLVa8HkkKymu4HsSzRaEBSkHODTMKecvruTQiZUuJggz3JU1n0jiUD4VSejBai9Z8pCclLSWSOYZJkykvEd4n+JPEftmPSbDN/EGkg7vD2HKOXS8EzuXSvtYYvzd38nI8BGRKnz8jrQ0VAEAnsnjizFiLb0n2a+Yj4mGvDMyP2c0XT06x98O1Xhqcd/krmoix+afbjnfWyP0+8myyulxDVCnYwhRTNGBi+GEn0AnH8VY19/Q9k+B4d22neG5z/8oiIvBfo516G5xYZf7XS26zq1svRV7BeUy/ytaQ3gdTQzbNygMjrTwU5t6K9LXpSIpByZKk5g7U362+w5T49WZ+/kp49qfNdw1plDmf3wP8Y1EheO0IUgO/aw8UM8uy2NO7Nd1n0vO3zQeRLLb7dUqipsc9PrDbBWvUWXQNxfXntsdyw9yeWKa8xU8JbHtJT4ABdDqFnr7BluYOpUFqaM/oMT6rLwVgrY/e/Sxm/N8GWu1c+y43jIwKwZZwxqjLnpW/KWA/VT2IXIq3sSI3Uc5Kfz9WkhSHKj+hT15DlyXOIWiN2ReghvfpZ8eApkGFD6wBr2VnDPpdv332li8zh3r28FWNxfUQZL1bFESKH50YiJWscYfGFbA+sY3yGGz3uknfPkclx81e+fcp9K8l3XIUnJftMolYLkdI+5YJoofBXXZouUGvJl7EwwFf73KIxBSzXkTN3+iBPrgtX95THmFjJR8jZsgV8l7xw6TL8aoXup9snkauVyfFZfhbZ+BVeia127GX3NbxGWiZyUhADwZwu63ZtsJ3ne4aIz3c/cLA39rZkBvb1065AarhaO1jv24V3I1LRvQrETVwF249ksZqEloQ84YqKHEJSPWcWIvJhmb/a9+kvsuJE35q/zuhlP8ELnSAy0I/Hg9r4IJL4zPXl2lGaBN916owjYYdXiDoUMt8lrVkHdcuZQf4zdZ0oVAmT0nvk+U47fJd/+0VMpl/5gp5GCUTe6n2IQviBVc0feHiX/UjSCnzmtbMPYgVCztis99U1hOF9eIulFiSIaFmeVpbjgh5am7MMRL+FPLFKWnyTzrgDcRKZvLPvbnzmeNiRXMtOPeOzvszJe8beX3+p2mVlDyFSLZFa62lFzoO9Yz/GJa8NmSG/xPz8meU3vCgq+fObvirJSZ6faZm0TT6okXWWhMyzuVfbn3ykXrArPYKXJ4iQOiF7HHrrQy5/fQ/e6AhNXm0dbMeLzjAPVl5L0z9wbCG7SurwjEe+Fs+AD68nt/b7ILXKS0I7MsrijFiVGvY95CmxZS9pkrEHPWS6/Zj6ARKdEGJVjpSrb3Pi21ORvW/QkEKWBgwcaaTfKaDAX/lBSUabhx+WXm3HDzI92QqI1tINsF9D7qCM02Ofz/P6mfwuEfXF2j5CRPsXNNJIpt/x9MfQIt1WOGLpiU20xV95AntAnH9k+a9eghWINNRL8EaHpmvVxBHLXsqiVbYjSzlX5jPb/Q3sp4x805NzQ8SkNTf0z5A7KNeOI1bk1U7jt3vCsSWcTR5FZlpeP2Ss/CgbZ7CMFXefbHk8e7RNw6O/8Qa+S+utMGo0PPVqJs5gDbMU1j6SKIfuB0mAJInG4zoVMnyDC4hev3og0i1Hm/yVZPyTDeWIbzMmIlJcExZ2iJQfOWRsZ931yVoUaZI6jrDCQrxiAaWlefnFgwiMlKwsAWuPDLFceYiwaI50n/kPcpfNz6+6F0Riy8NOZiRQyMtSkfhSCbcNnsUbzYsAc/d/iDaUybqFQmbNiaPa7pLaQjxhNJOS5K6HkHvvpPT2MG0NttPD6uNS+3WvzPkukdRqY8WSIXfUK0nju3ptknBYxuFVFFOFdzO6J+2wdtboy2tBYs9OiD0ZUMxg3y+72w/kit3sy/5Y0g+5CsTz7ozYmZWENctGCzLi214EGUj2AsPwicbwZCjmi0rruzLGHuJw6mWNIzakojpPVwl5Uv2R77IYU4LnO2bVMr2Un9lGMObYj7mL71Pj7dLOIyXkH3qS9xTR5K+uZ7cgwiBmeERQwajFnqueoo5aPogXOSdjzZkzT2GLepUeuTPrO5y+QYiRX7ft8hmNHaS0WWt6SGPOO6n5NQ9F3h7e7ie7ZI2ILHlWETEQIc+PC2EGxbGvsUarkBS0MCF1v3BrSPzVvTnZ5bveEI/CacpxgsVTwRjHiRLdjL2bcZIHXbm6Pbx5BrnE8XPYjavAc60n80WH/n/rzL6W9RCJq8iLE6LudXKJo7HwkOYfckPyQ7xEsm/nku3ccmf5WL3rR1YCdsmHaNbNBzkM41d3YEbMu5hXSBXcWIrGty9wlmpc3/BtlKg7e5KTwZZ9t5JZTMMr6rZHa3VtXTFotvNS8AjCH2K5hBexTzSsPpF36nnrpXfGToeNFeJA/HmYpz4dWiIhvQaOoa0Y5AawZdFDedBs2flGR6a37Efw4lD4CrteEqPwKzTuJWe+644uqcPIm0mzLuwK+dlcaXpkVWkBhFIOYk1ir62niQiwEJNBo16Vp+DV7aFb9FEiF6jO5IgcduqWJzqUWNdx+G0BJVNaNJ15ZXJ+2Zgm8863n3JO5UrxYimpT/bHnmsJftcTA2H+2Axxu39TgHgdowjuJ2RL3cH/EqeKYgmfX8kqa+z5jAyKHyYFsnovFvmMH8eo2M2Rt+zRMOgEIUNu9mA7x1boje3cX+V/PmO31Y51OiWU9njKb3opdTnRBsQDDAnZIzN67IA6YUaJT5hYlUKcHrOHcjuTdcSjhEwRHrBlDyb45j/7I/nthvmacXuNMNivGX+phIf9kU3ZDTnJM6WZ54U1n6lKWg1Wz8/QSzjAYKd+lGppRNY/67YCEU2Q1cV85SBJPeA/M0fpo88YZi2Kw3iUTKdsFc/DipvJhWz89nytyeBnIC93ETu2LD9Y6gdICcPaBgOR3N5SF7SNEJtrIsI5Sw61pMl2ij35TZgvr44UDPljQuSFdKxlIa2dAhs3i2ZGiwdfWqQ0vOAgEJGtFSP7I3+nM0Y9y/QsFGQaCBGjjoVvdwex4+SvEN+xO+yz+RlQ5HlqPmsSgcaI1ejnLbHLMKUOa2iIec7a98kF0e9ZRbQiM1Gn52GMwBVXr6jm/LR8RSETqtD4ZRdTfiUkquUtgk/d0tx7L/A0pxf8lBDhXc1L6XTknMzu+c+sYi376swTnp0Q+VsRuleITOxbbCcHMSBYNCE9N56wnlKF8zDG6MW2ZC/6/CB3XtjTqcn7aRciL6aNKNb0M4+TN+nMLtY9F6ze9Pp/41w+M6QWGG8R8rwgIp+5T7wAeZVTKmpmQ/RAyFuF2Vlz5LDbAT8UIgNWEG+ZcifeZE3COaRZD/Pr5vDE84f48xyWfN8Kv5JalzeM6jFTlsoP9BCR6Y4F+/jyMWtdhqo4ItjBE7kbEa3LiOicF3s+jTeGSL67rBy16HyiIA1n9uf6XSHCeV9J6+vwK1b1iprUfktry+uPAOmehALfXEh1icFK8cJLN2CH16+okMXnLK8bulf+JpKCh4WBPAuN+S1z+x0vC7xXf/qC4orbXj+uIk7ryftVJj4QScUC+3zSOesgm26eYhp7sNMpp0V6fPFXnqTD+r1y7O49K/IZkdoUqMOPqLrX2ALy1pQ/gbff2p1447v0pygbbcrd/dwDz9fLPA6Rd0iLeYGQBg9xii+Kf3EtW9bSKGBxnph2bCD6LcTrjSEXfdq4MZB5TltZ/7B7O+1secKIh09TD8sk/5G7cXegrJqd1XjbjhR4iCvBQ5xeRDZNzuBL+6eliDzp1s+v/EAa767S0glNji56+IamJlLCpVTVITDhKXUgm7L5jNpNJROx6R5XIrLKS7Ro4r3Ly03/f2TJlMtPwFeskDwej/jYkqNgIgsJyKyvdvRweVmYwZNuy8uty6veRFr1gBQQk2imD/L8ZgTs0S/xzLy4PyikSBAfn/FQV4c2Fm1Yq3GHd8XhV8xNIrecNpHPv6LeExJ2rldKzU8DLyJeLxvr3RGxasygREdslVlVK/0y1rF7slL3YjXgJCvJovSG83pCLHv9PiD7Sosl/sprAZCLrhz3kqeNd2Wv8r/jB7mjFez4+AlMe7xNRog0mSGDYsm/7zkfPuPyPTKfMT+eG9myzZt5FniJr/qxKPTHuWgNiBYurzc6HyKlHle6faAuzfKK3rsjIiSSZ6/x9pZV7vX0eLYsFTGojX2Z9Mbq90u+e/JqpkDi9jgb3l61JmS+4wexZFw7tfyOyPKZKj0buQZrO/KHoVeXF6yePKG/3IdbrOOx6k5+qIYtb6/QBm0s5MgxgHcj0e3LGvKjhOyQamPL0jTpPT4jT8ZTY/6QFme1BN9hqdFX4yEimSsDXv9qSyRhYtd1td/1Y4ktb88yhcexmnkFBbDK5RdJ5MgeejKdvEZ8afcDBp8eam3VOz9I3/IcwLVWn87C4Quvbs03FfErL/mpgc0f5A7e9ydlqB8WeDceZD9iW0T2SpmnLbzgZo8fTTuj5zaC2yzxVxk1ypjcyONHQIF4/aGJKhBr7pw6q6wIqWNl7PQtEQVr9nnGD//z5rU1XxWDBaddK6w9Nnjm8kBO5R69Kz6tJsTHZIVyHQkVMKSuo2QRnq/fP9qWYS9prR795jm27AXzy+bb922T+aJrneoXuLEdORP3ft7++q+C1R/il4q0h9jg8qHQGGJ2dgtxLurw3eRMFEQL15bv62sFyBJ75m1Ny6MCcWIvSYhnf2PHcJ3UzmEUdJ2SXmEFuXW6bx5Rk5yZyuc0wTp+AeHCrpAQ0cqFSj5LNEoWhFLncePBXXIR6vAr/g8k+r1U1Oq3yjwxPqZWp0cPiIi1TGa0ruv1d1oh4plXvF9myXhoCrkGryTzPo7zfae+gF0qIa8U3jPiJZJFGuFJSTtOL7WGeZezqjkltzH5jGLmROb03DiMs2ni9+Q4ez2nMGhP7dT0tV92mwdFIIemhbxZYU/IbS0hp2I9Lz770fxip/nj6wk5b2/q1Vf8AorEX8nit4Vdj+U3gkberLreiE7IMPJPvOmdzXZufqNSwsU8pSXItTQV4ROPEvJOYxxphyAlb/DEhfQSyVdlqrJcFeQebFHwbQO5jvvHYAMyALdYuOwppEWuXvQoHttZ8kqYQbFdf3tOLRE/hXDZwytjwXMBWyK4xyh815teBZ39ke8ZC+Ki26/c8wRoIMlKbodIE/U8kMwtBZX3AtMT8vwiXIyGWHArNviuYT3y/ODWWGR9bAOyvD4SVuX2i45EkzBiXjRInlsk0vIm694p+F0oHEMPyMgKE5Gm9Rs+gYxaRAMekacpg/1ynZVz3XxGSryQb2yP01qG9ZRdjJJUxCqFuAuGOLYQPxgA+y7EryPDftz2kJFWBt9V5a9WxI2FrK6VgHHO7W55E/xVH1JKiW/vUwQRq1tLu4srI39j5/30ofDxd5YC8lQrIK+7nmLLzz1xsNytNXmMGS+7eF3yjxyW+OSLwzf3+qxedQbvKlVLeXFUS1uy1OxhGVIA57GdGd/nvmwhywOsGPkiTfapWOvXzBa5RJjTIk8mhE/Lb888oPm33+FsF0xm1yShZ81q8ekc7uddYuZhcR92VzHRw3NJu+rLHk/Q7Gprts+XVr9tfYKH76blXQe4n1aXF8aDJdptLE+/5K+GhCG19EH8OnPIhjR67dxn3N137A4iJ57lvbW8P4hMN+/b2iIXaW7qzO65IdzH/9npRI9+683Ty94AOXdIS7Dlq9UeOMsj5Bh6JlKTu914+/CIOSNCUjRZwoCdESFlBuZ9yQb5HZxgjELkqLAOlSxVuZ579YdML2rEm4Y8PTJYBdMT4m4I18Ws4n6DekNM0FNuIHWzifvxrOv2KjCiAImIvDTGV/f0wlSP+nnKeG6eCNseOvEipECyLDNrmPhRxTASznbtpeFIGcxz+/2xLQ8+Y3KpN/yvvf1OHkP8cIuYt/lZy9v9rwqvxAum1cTbkbY+1W8w4K88HMSzOXuf4XVx+CtzR4k603MzxNUxO0fUIT6w931ilruDevj7pOwW5IPMuwfl0EtE+6FZIiI7qfOZfkQRkUu8zzQPp0A2zipilpU93N0Pc/JX9sup/yDv5zj9Ide3ri95y/2lvZI13WhikGznZi+USo10q73ByhXyo+SvGzXbFVlPPROxtHKm5hcBuNPgpwjx+5YpqyKdJbHmw5ZNLrNXPpOfFw1Ay1ZS94rYREoXXeczU2Q1YsdHoiETx/x5r8/YI9mpkN+P2M7zGvHU/C8cL/aDHr7oNeBpC35XzmWw5e3Z1zZom3wHOn50uDRLOLyhXnOerX+4hJ/MmTwbKGS1s6hJ3tM/VpX0imD1Mqf0hDSTn2UFksWrWEVWrmeftrET4c6o/FzsLp3gBbMb4pknagFqvvBMTEe6FxEPLTi/pJQ99JwBS9B1xy/i/cize8ZXrH8B8d2CDc/leHTci54C8ZpFCbrOT0XOuzDOx6+U8+q3RFzdIJfmpHTkAMGmCLHlV10D8RJzjPP71TpibPxSv0rvU0tZDv2MaSA26LccFQ8pEBGPOshMO54znakzj982XDLsqezk25m1Po7WaBIbxXfl8o4f4gHSm0eS2M6QoLJirZCXQ4fMnzzT9hqNQJZYP/fahIjqMfJ/8jm+98B2fI7T4q/uCfVBQx4vmVgeJUrfLecGWuJoAH91Q4ikcQusw/GQyCiwlZo9+e/p8ldLriZ3hYSs7EVCiEhzFMph8XqIATb31CLiwpqExwvYjQV/+dQ6QuJJ0lPFljsrwEtn/c6eYHz8fvrYPi1PkZmKOPbxaOtjBd1Tn5/cROREyNIKpMy3EIIkD7PTNKUaQsh889tABrS6BPWXVJaISNPzNsbThnM2yrw+0iJvDHEa0z+1gk/7Z6IK3+VHTQ7iLacH8xpOGNXuJ9l5I48Qj6AFPrOv76Dg7f3dI0KGd40g1TvBGM/woL5R+w2/K8aw63pGPUceF9bFkJsbI87LnDEkqYw+Cekz8XZaJ4K7l8JnllvBwmcsNSNjFCKtURFPOL7roE9FO1NfWQe8dSFyw3kS50x5xyLHbEdMTxp7EOmycpfP5BtG/Ly9iIR05P8IkVzyPMiZo/tmBFv28mOZ4zPlbRnvuznTd4o69c88NyXusAh5mkTEaX3LfPtxKiAlh0k+78fRJPRcy2tcP8aGNbjWlYNBDblkdeOhjMlfTDWCeeoj12msBHV2kf7biOmJ7+caO2o7H9H3uVLgMye9yXs0jhOOV8Cxj0y1b1VDVmWYRegpUX5H82VOu0RXOqEgP0oLp4TIDFIhXrCfmkRqIgbW0hEya2BM+Jzue4Y4IStkpsDcOa9E6SWS8F0S39B4MugceWnrIJLs97q3z/1fQu6VRsaKkxCGvdjy9WgGT6Oc69du02M9t0qVRHgB53b5jPnx7X5acdzPM5KNnNnOcENDObxa/495F8dvW5+seCPkiI59vnRnz0afRPyEauOXHum11vkuWeV1Pu96rU1GabSy9f5KvmHl1teRU+H1WLtvUACZMk8VdVfU4XLzh9t4bvMnj/rIdu7K81ZCzo68Yep4yfrOLILjxzjO91cvdz+P94c8yUoMyDM/L76cC3y988pIOWG/8jy/X4q3/8gIjS0fEev9eTLWRDT1vOM7Cog6OqLh+LSjLotNYHye72mWzHbsen2xv+/yDZbbDV7/DaJN+tQNpMjgL8zyDSv4lQ9EPOGXOXhXdMRrbbOdG6wOMAchfhlBYH/ukD8GJnw9WeJxLm7UOL+LyPaVTX7GjPobpdceM+GvZ36XAUatZSrDxHpENw6/+Tby7VOsYCCSfOM6ziz5K5P5juGD+Koc7KFHHCbOvl2vl9151vV3Sdbl3XlXxMaDTx/EKwpDwqX2lwYEbNkvs5MlvPxVCzPx3pwrnS5HGztHUkgWEmMO14+EypMj0rtc70NExvJFyNjNZ9dXYLuvJ2zHiKjaleMSZJzzBymbu5y3eFrjASNyJIl0JyJze0geSGnNnXoi7l9Q5mVM4wiLLXvJ6jr5K5mHwRtjb7lyFFhh2I+ntHUHW35yCB+4+q1pLvnDlcjO1XAm4tbu22TQP1cOaw2shHD9QHzsn2eW1gkreN+6rxYK8uJuPX69J6KFUrPFesH+4G1B7uqgtDS/QamALV/fCawZebm35ZQyd/9vq6Fu3nsrRM555OqW8ln7gcFeL1xoZKe3SdU21vu63euCbuwXCLHuQQcgvyL1iNfdnkLuD5nw18v0fGyuEJv7o+tkYUZhBZ7bvfbb4YrrXi2XNTpu96MEDxHOq6lKIiH8ivWLhrGHv5PskT3UVBzWORHyuugpxrBb1jrEWRhpzPASKycLOdJH1CQj+hYvdaakWyYevrmQ6b3El44W7mIGoKS9DxFbtjMl0ryt4A4pltjhO9zx1FSCp3mn37/Fs0t31uUVxQeRM2+mfpYCXYvnPR154mwYjZWDWCyijnd5+I75P36kt65KaVmeKDtRHe6uLSpq2B8UIu/cPu24z8FKs3dZ9vPBkExJnB+0u0SczFAnbK0CzxQG4l3mXpLIl3vHiHneLTa/eM/a3V5iaiAuIc4gtdGpV3+BiQym58jxIsxEPAuYekMu9JZqxXe5c7V4Qv+6m9QjooVXpuBIvaCHx+MZExG869d01kgJd13sZQmBOCcZi89U81LpmAunMTFTou7SamaM6F6TBbNC5Ml9ntj3vHLUT+C9MF5aPVuHF3ktVosLGYlCRMuZV3m9Ll9kPpKbdi/gHYh4RWqyAg9sN95Oe01ewBqNPex+/IxfaqOels8HmdEvvgOi/h6evb22Q26s5CNExpK3xgsZ+xXqMS+aEU7iu6yNy7qyQtbLi7byeRrGpdZ68qc/0afrG8VrI17nhcCDFAdalkMkIwuvxE8473G/yBmXWYtX5HDmhsiAEL3+0n49EdgYD3v4/Hph7GCa/MNdc/ogXawa+aIW5DZ56UQgxbNrMBrymXZLjI9ZkKl6cRMZqbWAuhmegyuFA9kwOcKiBahp4Ke7e0psR6LqtbMukFG8LPIEMr1STmXLO/p04e3Rt5JZW1XSvbtRWoTYrP0SEdPbPCdu4qpl0tO0FJbWP7x1ad3n9gC/8gpzqSPWbR5498s5gWgBVp5CEnLH4p3IlrZn/3Q+80LQp3+QFT/RMBPZ8bPrH6R7wiqRPJIcVoxqLmW0hXVhuepPVvwTcr1wxQEyx2i5EBFnyqy3Y3lLDPanHc3Y5t3BQvywKbSW9F4u/SI3VYjfaImovp9YqyJXfMY9jgKrZ142sFVE/q1UmSHGbcyrP4vtHCK25hxsxy+m430l5rd6yIfGGBZRlM76G1aO9MS8fEZeox8PIpKjVzsDYiLh3CeyogVvzIqRJO8aeNZe3qrfg4eImZfUzneDs0mzrOJb50C8uGhENp35tQfF4CWZ397mWW9/SAsl7c/qbin0y4xEc/5ceN+6+dV1h6cFhQyNNZieELn9zDu11p8rSXxpG91TpjA+v/uGef+yiWpFP1MN5EiPDESorAcPvMFPMakjaUxEFM0zop9xDLs4S9ngvdbldYcFniDEZHWQL2r92PXrPoDcfrwsxx8y5EVm1hazIbdErjd6OCRzldUtbOwYy6XemFoncksOEY37pZ6ffk05Y5U2k4xuRJ6eTb97lmxZpNPrdyJ26vTobeOXzjXz4KkE34L+3ssgRnnE9aij/IYMY+RWSPUatpVI97Aon5G96vHTH3fNeU+WrSi+urne/TrUwEwMk99Z9rx8RopkJpw4MDHz4EdUPsjcnfPuNdHvw86aEN+n6XzmJlm02z+I380AeV42vEB/IaJxL9S9O0y9H/latpPX52aft9+/2ilRnhwmMoH+eJrX5W67kNY7zzx6YQYRuU/LK0tB8iu214FaOFdivin+uStBRNkv6UFWlZbJsZqQe2nHNVvEaUrzjb0Z+PbjeUWMTJqo8+9czR9y5QLswi+9Xq1pUP/I0+t+TQ6R6ZWVAhG7l6cF7foZ/UXNJksg0ob8Z7s7eCB7EMnvVZzJsnuehxjwXdfv7eJethDz6haYd3H+PjeiIiaS1CLvJbcfcWm0BZoJLyP0iMgDY6RU1Dj5Hcx8ZixJITwFL63u2oYt7yO/iXZZE3ZXQPTbJO5+EyfbuTOniD0OIfuGM4lo/HqAF2m+v+J3cvwhb2x3ovGMRy69JDQQyapn+AHZd8rUbCJOERHlM5kYOb60wu+8eC81v4xyiI0c+/1qTGGc3+/ofUFs5wVpJA1SA5L9RnZkP3qh5x1ZVcDHIn7uxn1BfTkXulfIke6HJnkxvJaY4f/iEH/f94vskaBbhAwvVcpn5IevAhkT8rxsPfoTJXexY6XIBIcgQQOS4s21wJ5KW/pltODqngk6ake08CVp48g9sidSLm/r8JlR866IEb0k6ieB+pO6lz0kPVAtwTdB2idm7rUk/PQJWpZW659zWy/fI18U7PRlUycnNLbfsLITTxOImHrFc9idJ0Ype4b95SeelY3rQkhKbtWAVFsp42zXK734hdBEpMfGhdV7ZZpX3sUYiq+uzy32zw9o7M1xrkXmYyNf6zk5HBfxjfcr1tQbnxHXmAE6yu+cC8a76vzWLM+p56809o/3IQrxmxKgAZ68TBFmZKr4GUSvpN+B+GcFeAFCbBze3/Sap3TwLIOQ7NYTIy86Yn5ZA5BRQzbskr+2fmcrMDteBbjxnojXbLcxkb32erjFI0BAJE6Td4gI2Z45RGSN1nhC5PXT/apFtmyefYA4wPPc0NixM/JGvvNcrp2hRjLvRnlSNrKnuKHs+UX3gadEn3SEfGZK+PCzm4a6u37hbmq8PUqT/rtqA+Ps5VEmazs/Px9WmMkspT6SJyECmb77jjyH52zsjMVfyTYUnl/2UkzDz8gBkTlPAVH0t8J4nkwDRAoqMIvp+aHrFhAp9YREWQdqCXUvhAnW/dbq+TEb8/lZwVy5lte+8oSN73pXFoUStT1RL3KWd6rJePvh8wOOmb6VEK+lAfsuRDyXd44/EeUeJleTx3v9JgYg59zEWqYyJ14QBNGVd8LVugQbfCddv2Xgg8gRoKfwTmmr8s5xLwmoZcG5kE32+rxE/MzuoRwem34PBHooalFTpK67ceQcEVV7NwURTdpc9dfrg0OirlSksf7G72aUT733d1cJxrp874oXHOYVvCvCIQnHqFqKq380pOWswYCf+7xVcdFLxM80Iyf5ybWrkacJnuTijcq3+8XBn5xkdU/vYuT/2Q1dbhrb8RKgrMT7nOo11lL2Q1xnsgqEnzB8ixXJxIKltni71nty19ddbKeZR4nRZ6mAuyq1um9gGm9W/X+Eq9n007AAAA=='

In [8]:
SPLIT_MD5 = {'train': '34976288777450b702c2f7913271d95f', 'val': 'c8393491cb9d4952194cdbf7f459d3a6'}

def load_frozen(b64, name):
    raw = gzip.decompress(base64.b64decode(b64))
    got = hashlib.md5(raw).hexdigest()
    assert got == SPLIT_MD5[name], f'{name} md5 {got} != {SPLIT_MD5[name]}'
    return pd.read_csv(io.BytesIO(raw))

split_tr = load_frozen(SPLIT_TRAIN_B64, 'train')
split_va = load_frozen(SPLIT_VAL_B64, 'val')
train_ids = split_tr.image_id.tolist()
val_ids   = split_va.image_id.tolist()
print(f'frozen split loaded: train {len(train_ids)} / val {len(val_ids)} | md5 OK')
assert not (set(train_ids) & set(val_ids)), 'train/val overlap'
assert len(train_ids) == 10054 and len(val_ids) == 2514

# reproducibility check: does phase 0's recipe still produce this exact split
from sklearn.model_selection import train_test_split
import sklearn
strat = index.has_defect.astype(str) + '_' + index.primary_class.astype(str)
strat = strat.where(strat.map(strat.value_counts()) >= 2, 'rare')
a_, b_ = train_test_split(index.image_id, test_size=0.2, random_state=SEED, stratify=strat)
same = sorted(a_) == train_ids and sorted(b_) == val_ids
print(f'regenerated with sklearn {sklearn.__version__}: '
      f'{"identical to the committed split" if same else "DIFFERENT - using the committed one"}')

idx = index.set_index('image_id')
LBL = idx[[f'has_class_{c}' for c in range(1, 5)]].astype('float32')
for nm, ids in (('train', train_ids), ('val', val_ids)):
    sub = idx.loc[ids]
    print(f'  {nm:5s} defect {100 * sub.has_defect.mean():.2f}%  ' +
          '  '.join(f'c{c}={int(sub[f"has_class_{c}"].sum())}' for c in range(1, 5)))

frozen split loaded: train 10054 / val 2514 | md5 OK
regenerated with sklearn 1.6.1: identical to the committed split
  train defect 53.03%  c1=718  c2=200  c3=4116  c4=644
  val   defect 53.06%  c1=179  c2=47  c3=1034  c4=157


### 5. Decode once into a memmap

Six training runs over 12,568 JPEGs would decode the same 1600x256 files about seventy-five
thousand times. Decoding once into a uint8 memmap at the training resolution costs one pass
and about 2.6 GB on local disk, and every epoch after that reads from the page cache. This is
the single biggest speedup available here and it changes no result.

In [9]:
from concurrent.futures import ThreadPoolExecutor

CACHE_PATH = f'{WORK}/cache_{IMG_H}x{IMG_W}.u8'
all_ids = index.image_id.tolist()
ROW = {iid: i for i, iid in enumerate(all_ids)}
shape = (len(all_ids), IMG_H, IMG_W)

if not os.path.exists(CACHE_PATH) or os.path.getsize(CACHE_PATH) != int(np.prod(shape)):
    t0 = time.time()
    cache = np.memmap(CACHE_PATH, dtype=np.uint8, mode='w+', shape=shape)
    def fill(i):
        im = cv2.imread(f'{TRAIN_IMG}/{all_ids[i]}', cv2.IMREAD_GRAYSCALE)
        cache[i] = cv2.resize(im, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(fill, range(len(all_ids))))
    cache.flush(); del cache
    print('decoded %d images in %.0fs' % (len(all_ids), time.time() - t0))

CACHE = np.memmap(CACHE_PATH, dtype=np.uint8, mode='r', shape=shape)
print('cache ready: %s, %.2f GB' % (str(shape), os.path.getsize(CACHE_PATH) / 1e9))

decoded 12568 images in 41s
cache ready: (12568, 256, 800), 2.57 GB


### 6. Dataset and augmentation

**Grayscale into a 3-channel backbone.** The images are 1-channel, ImageNet weights are
3-channel. We replicate the channel three times rather than averaging the first conv across
its input channels. On a single-channel input the two are equivalent up to a constant scale
factor, but replication leaves the pretrained first-layer filters seeing exactly the input
statistics they were trained on, and it does not throw away the color-opponent structure the
first layer learned, which still contributes useful oriented-edge responses. It also costs
nothing to implement and keeps both backbones byte-identical in their stems, which matters
because the whole point here is that only the architecture differs.

**Augmentation** is horizontal flip, vertical flip and brightness/contrast jitter, and
nothing else. Rolled steel is close to flip-symmetric so those are label-preserving. Strong
geometric distortion is deliberately excluded: class 2 is a thin vertical line and class 1 is
a small spot cluster, and elastic or large-rotation transforms bend exactly the shape cues
that separate them.

In [10]:
TRAIN_TF = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
VAL_TF = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


class SteelCls(torch.utils.data.Dataset):
    '''Returns (3xHxW float tensor, label vector). `binary=True` collapses to has-defect.'''

    def __init__(self, image_ids, transform, binary=False):
        self.ids = list(image_ids)
        self.rows = np.array([ROW[i] for i in self.ids])
        self.tf = transform
        self.binary = binary
        y = LBL.loc[self.ids].values.astype('float32')
        self.y = y.max(1, keepdims=True) if binary else y

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        g = np.asarray(CACHE[self.rows[i]])
        img = np.repeat(g[:, :, None], 3, axis=2)     # channel replication
        return self.tf(image=img)['image'], torch.from_numpy(self.y[i])


def loaders(binary=False, batch=16, workers=None):
    workers = workers if workers is not None else min(4, os.cpu_count() or 2)
    tr = SteelCls(train_ids, TRAIN_TF, binary)
    va = SteelCls(val_ids, VAL_TF, binary)
    g = torch.Generator(); g.manual_seed(SEED)
    return (torch.utils.data.DataLoader(tr, batch_size=batch, shuffle=True, generator=g,
                                        num_workers=workers, pin_memory=True, drop_last=True,
                                        persistent_workers=workers > 0),
            torch.utils.data.DataLoader(va, batch_size=batch * 2, shuffle=False,
                                        num_workers=workers, pin_memory=True,
                                        persistent_workers=workers > 0))

_ds = SteelCls(val_ids, VAL_TF)
_x, _y = _ds[0]
print('sample tensor', tuple(_x.shape), _x.dtype, '| label', _y.numpy())
print('pixel range after normalize: %.2f .. %.2f' % (_x.min(), _x.max()))

sample tensor (3, 256, 800) torch.float32 | label [1. 0. 0. 0.]
pixel range after normalize: -1.67 .. 2.64


### 7. Models, losses and the training loop

Both backbones come from torchvision with ImageNet weights and get a fresh linear head of the
right width. Everything else in the loop is shared, so a difference in the results is a
difference between ResNet-50 and EfficientNet-B2 and not between two training setups.

On the imbalance, we compare three settings on the champion: plain BCE as a control,
class-weighted BCE through `pos_weight`, and focal loss. `WeightedRandomSampler` is
deliberately not used, and the reason is specific to this dataset: it assigns one weight per
*image*, but 427 of these images carry several defect classes at once, so no single per-image
weight can rebalance all four classes at the same time. Oversampling for the sake of class 2
would drag along whatever else those images contain. Loss-level reweighting acts per label
and does not have that problem.

`pos_weight` is capped at 20. The raw negative-to-positive ratio for class 2 is about 49, and
letting 200 training images carry that much of the gradient makes the run unstable for a gain
that shows up in nothing but that one class.

In [11]:
def build_model(arch, n_out):
    if arch == 'resnet50':
        m = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, n_out)
    elif arch == 'effnetb2':
        m = torchvision.models.efficientnet_b2(
            weights=torchvision.models.EfficientNet_B2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_out)
    else:
        raise ValueError(arch)
    return m.to(DEV)


class FocalBCE(nn.Module):
    '''Multi-label focal loss (RetinaNet form, applied per label).'''

    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * y + (1 - p) * (1 - y)
        a_t = self.alpha * y + (1 - self.alpha) * (1 - y)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


def make_loss(kind, y_train):
    if kind == 'bce':
        return nn.BCEWithLogitsLoss(), None
    if kind == 'posweight':
        pos = y_train.sum(0)
        w = np.clip((len(y_train) - pos) / np.maximum(pos, 1), 1.0, 20.0)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w, dtype=torch.float32,
                                                           device=DEV)), w
    if kind == 'focal':
        return FocalBCE(), None
    raise ValueError(kind)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast('cuda', enabled=DEV == 'cuda'):
            P.append(torch.sigmoid(model(x)).float().cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def macro_f1(y, p, thr=0.5):
    from sklearn.metrics import f1_score
    return f1_score(y, (p >= thr).astype(int), average='macro', zero_division=0)


def train_model(arch, tag, loss_kind='bce', epochs=12, batch=16, lr=2e-4,
                binary=False, patience=3, log_every=200):
    '''Train one model. Returns history + the best-epoch validation probabilities.'''
    set_seed(SEED)
    n_out = 1 if binary else N_CLASSES
    dl_tr, dl_va = loaders(binary=binary, batch=batch)
    y_train = dl_tr.dataset.y
    model = build_model(arch, n_out)
    crit, w = make_loss(loss_kind, y_train)
    if w is not None:
        print('pos_weight:', np.round(w, 2))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')

    ckpt = f'{OUT}/ckpt/{tag}.pth'
    hist, best, best_ep, bad, best_p = [], -1.0, -1, 0, None
    print(f'=== {tag} | {arch} | loss={loss_kind} | {epochs} epochs | batch={batch} ===')
    t_start = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        t0, run = time.time(), 0.0
        for i, (x, y) in enumerate(dl_tr):
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=DEV == 'cuda'):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            run += loss.item()
            if log_every and (i + 1) % log_every == 0:
                print(f'  ep{ep} step {i + 1}/{len(dl_tr)} loss {run / (i + 1):.4f}',
                      flush=True)
        sched.step()
        p, yv = predict(model, dl_va)
        f1 = macro_f1(yv, p)
        hist.append(dict(epoch=ep, train_loss=run / len(dl_tr), val_macro_f1=float(f1),
                         lr=sched.get_last_lr()[0], secs=time.time() - t0))
        flag = ''
        if f1 > best:
            best, best_ep, bad, best_p = f1, ep, 0, p
            torch.save(model.state_dict(), ckpt)
            flag = ' *best'
        else:
            bad += 1

        # Survive a disconnect. This run is several times longer than Colab's session
        # stability window, so every epoch's state goes to disk instead of living only in
        # memory. A dropped kernel then costs one epoch, not the whole run.
        with open(f'{OUT}/metrics/hist_{tag}.json', 'w') as fh:
            json.dump(dict(tag=tag, arch=arch, loss=loss_kind, epochs_run=len(hist),
                           best_epoch=best_ep, best_val_macro_f1=float(best),
                           history=hist), fh, indent=2)
        if flag:
            np.save(f'{OUT}/metrics/probs_{tag}.npy', best_p)
            np.save(f'{OUT}/metrics/labels_val.npy', yv)
        print(f'ep {ep:2d}/{epochs}  loss {run / len(dl_tr):.4f}  '
              f'val macro-F1 {f1:.4f}  {time.time() - t0:.0f}s{flag}', flush=True)
        if bad >= patience:
            print(f'early stop: no improvement for {patience} epochs')
            break
    del model; torch.cuda.empty_cache()
    print(f'best val macro-F1 {best:.4f} at epoch {best_ep} | '
          f'total {(time.time() - t_start) / 60:.1f} min')
    return dict(tag=tag, arch=arch, loss=loss_kind, epochs_run=len(hist), batch=batch, lr=lr,
                best_epoch=best_ep, best_val_macro_f1=float(best), history=hist,
                ckpt=ckpt), best_p, yv

### 8. Smoke test — one forward/backward per architecture before committing GPU hours

In [12]:
BATCH = 16

def smoke(arch, n_out=N_CLASSES, batch=BATCH):
    set_seed()
    torch.cuda.reset_peak_memory_stats() if DEV == 'cuda' else None
    m = build_model(arch, n_out)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')
    ds = SteelCls(train_ids[:batch], TRAIN_TF)
    x = torch.stack([ds[i][0] for i in range(batch)]).to(DEV)
    y = torch.stack([ds[i][1] for i in range(batch)]).to(DEV)
    t0 = time.time()
    for _ in range(3):
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=DEV == 'cuda'):
            out = m(x); loss = F.binary_cross_entropy_with_logits(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize() if DEV == 'cuda' else None
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEV == 'cuda' else 0
    print(f'{arch:10s} out {tuple(out.shape)}  loss {loss.item():.4f}  '
          f'{(time.time() - t0) / 3:.2f}s/step  peak {peak:.1f} GB')
    del m, opt, x, y; torch.cuda.empty_cache()
    return peak

p1 = smoke('resnet50')
p2 = smoke('effnetb2')
print(f'\nbatch {BATCH} at {IMG_H}x{IMG_W} fits both backbones' if max(p1, p2) < 13
      else f'\nWARNING: peak {max(p1, p2):.1f} GB is close to the limit, lower BATCH')
print('smoke test: OK')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 146MB/s]


resnet50   out (16, 4)  loss 0.6256  0.77s/step  peak 3.3 GB
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 166MB/s]


effnetb2   out (16, 4)  loss 0.6394  5.60s/step  peak 4.4 GB

batch 16 at 256x800 fits both backbones
smoke test: OK


## 9. Baseline — binary has-defect

The first question is the easy one: is there a defect at all. At 53/47 this is a balanced
problem, so it gets plain BCE and no reweighting, and it sets the floor that the four-class
models have to be read against.

In [14]:
res_bin, prob_bin, y_bin = train_model('resnet50', 'binary_resnet50', loss_kind='bce',
                                       epochs=6, batch=BATCH, binary=True, patience=2)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
yb, pb = y_bin.ravel(), prob_bin.ravel()
res_bin['metrics'] = dict(
    accuracy=float(accuracy_score(yb, pb >= 0.5)),
    f1=float(f1_score(yb, pb >= 0.5)),
    roc_auc=float(roc_auc_score(yb, pb)),
    pr_auc=float(average_precision_score(yb, pb)))
print('\nbinary has-defect on the frozen val set:')
for k, v in res_bin['metrics'].items():
    print(f'  {k:9s} {v:.4f}')

=== binary_resnet50 | resnet50 | loss=bce | 6 epochs | batch=16 ===
  ep1 step 200/628 loss 0.3734
  ep1 step 400/628 loss 0.3174
  ep1 step 600/628 loss 0.2873
ep  1/6  loss 0.2865  val macro-F1 0.9220  175s *best
  ep2 step 200/628 loss 0.1738
  ep2 step 400/628 loss 0.1792
  ep2 step 600/628 loss 0.1775
ep  2/6  loss 0.1783  val macro-F1 0.9451  172s *best
  ep3 step 200/628 loss 0.1228
  ep3 step 400/628 loss 0.1303
  ep3 step 600/628 loss 0.1348
ep  3/6  loss 0.1334  val macro-F1 0.9554  171s *best
  ep4 step 200/628 loss 0.0991
  ep4 step 400/628 loss 0.0976
  ep4 step 600/628 loss 0.0947
ep  4/6  loss 0.0929  val macro-F1 0.9629  171s *best
  ep5 step 200/628 loss 0.0620
  ep5 step 400/628 loss 0.0677
  ep5 step 600/628 loss 0.0668
ep  5/6  loss 0.0676  val macro-F1 0.9701  170s *best
  ep6 step 200/628 loss 0.0471
  ep6 step 400/628 loss 0.0486
  ep6 step 600/628 loss 0.0459
ep  6/6  loss 0.0457  val macro-F1 0.9689  170s
best val macro-F1 0.9701 at epoch 5 | total 17.1 min

bi

## 10. Which imbalance treatment

Three short runs on the champion backbone, identical apart from the loss. Five epochs each,
which is enough to rank them without spending the GPU budget three times over. The winner is
what both full runs use.

In [15]:
ABL = {}
ABL_PROBS = {}
for kind in ['bce', 'posweight', 'focal']:
    r, p, yv = train_model('resnet50', f'abl_{kind}', loss_kind=kind,
                           epochs=5, batch=BATCH, patience=5, log_every=0)
    ABL[kind] = r; ABL_PROBS[kind] = p
    print()

from sklearn.metrics import f1_score
print('imbalance treatment comparison (frozen val set, threshold 0.5)')
print(f'{"loss":12s} {"macro-F1":>9s} ' +
      ' '.join(f'{"F1 c" + str(c):>7s}' for c in range(1, 5)))
for kind, r in ABL.items():
    pc = f1_score(yv, (ABL_PROBS[kind] >= 0.5).astype(int), average=None, zero_division=0)
    print(f'{kind:12s} {r["best_val_macro_f1"]:9.4f} ' +
          ' '.join(f'{v:7.4f}' for v in pc))
BEST_LOSS = max(ABL, key=lambda k: ABL[k]['best_val_macro_f1'])
print(f'\nchosen for both full runs: {BEST_LOSS}')

=== abl_bce | resnet50 | loss=bce | 5 epochs | batch=16 ===
ep  1/5  loss 0.1731  val macro-F1 0.5948  171s *best
ep  2/5  loss 0.1072  val macro-F1 0.6333  172s *best
ep  3/5  loss 0.0811  val macro-F1 0.8146  172s *best
ep  4/5  loss 0.0582  val macro-F1 0.8978  171s *best
ep  5/5  loss 0.0465  val macro-F1 0.8924  171s
best val macro-F1 0.8978 at epoch 4 | total 14.3 min

pos_weight: [13.   20.    1.44 14.61]
=== abl_posweight | resnet50 | loss=posweight | 5 epochs | batch=16 ===
ep  1/5  loss 0.5629  val macro-F1 0.6393  171s *best
ep  2/5  loss 0.3364  val macro-F1 0.5836  170s
ep  3/5  loss 0.2602  val macro-F1 0.6868  171s *best
ep  4/5  loss 0.1825  val macro-F1 0.7348  171s *best
ep  5/5  loss 0.1479  val macro-F1 0.7795  171s *best
best val macro-F1 0.7795 at epoch 5 | total 14.2 min

=== abl_focal | resnet50 | loss=focal | 5 epochs | batch=16 ===
ep  1/5  loss 0.0182  val macro-F1 0.4396  172s *best
ep  2/5  loss 0.0114  val macro-F1 0.6942  170s *best
ep  3/5  loss 0.0089  

## 11. Champion — ResNet-50

In [16]:
res_r50, prob_r50, y_val = train_model('resnet50', 'cls_resnet50', loss_kind=BEST_LOSS,
                                       epochs=12, batch=BATCH, patience=3)
np.save(f'{OUT}/metrics/val_probs_resnet50.npy', prob_r50)
np.save(f'{OUT}/metrics/val_labels.npy', y_val)

=== cls_resnet50 | resnet50 | loss=bce | 12 epochs | batch=16 ===
  ep1 step 200/628 loss 0.2360
  ep1 step 400/628 loss 0.1949
  ep1 step 600/628 loss 0.1754
ep  1/12  loss 0.1731  val macro-F1 0.5948  171s *best
  ep2 step 200/628 loss 0.1135
  ep2 step 400/628 loss 0.1109
  ep2 step 600/628 loss 0.1080
ep  2/12  loss 0.1090  val macro-F1 0.6593  170s *best
  ep3 step 200/628 loss 0.0887
  ep3 step 400/628 loss 0.0901
  ep3 step 600/628 loss 0.0912
ep  3/12  loss 0.0913  val macro-F1 0.8168  170s *best
  ep4 step 200/628 loss 0.0683
  ep4 step 400/628 loss 0.0734
  ep4 step 600/628 loss 0.0752
ep  4/12  loss 0.0749  val macro-F1 0.8179  170s *best
  ep5 step 200/628 loss 0.0603
  ep5 step 400/628 loss 0.0636
  ep5 step 600/628 loss 0.0654
ep  5/12  loss 0.0657  val macro-F1 0.9037  170s *best
  ep6 step 200/628 loss 0.0529
  ep6 step 400/628 loss 0.0496
  ep6 step 600/628 loss 0.0514
ep  6/12  loss 0.0518  val macro-F1 0.8348  172s
  ep7 step 200/628 loss 0.0409
  ep7 step 400/628 lo

## 12. Challenger — EfficientNet-B2

Same input size, same frozen split, same augmentation, same loss, same optimiser, same
schedule, same seed. Only the backbone changes.

In [17]:
res_b2, prob_b2, y_val2 = train_model('effnetb2', 'cls_effnetb2', loss_kind=BEST_LOSS,
                                      epochs=12, batch=BATCH, patience=3)
np.save(f'{OUT}/metrics/val_probs_effnetb2.npy', prob_b2)
assert (y_val2 == y_val).all(), 'the two models saw different validation labels'
print('both models scored on the identical frozen validation set: OK')

=== cls_effnetb2 | effnetb2 | loss=bce | 12 epochs | batch=16 ===
  ep1 step 200/628 loss 0.2213
  ep1 step 400/628 loss 0.1739
  ep1 step 600/628 loss 0.1509
ep  1/12  loss 0.1486  val macro-F1 0.8148  175s *best
  ep2 step 200/628 loss 0.0845
  ep2 step 400/628 loss 0.0843
  ep2 step 600/628 loss 0.0821
ep  2/12  loss 0.0831  val macro-F1 0.8176  165s *best
  ep3 step 200/628 loss 0.0602
  ep3 step 400/628 loss 0.0638
  ep3 step 600/628 loss 0.0658
ep  3/12  loss 0.0659  val macro-F1 0.8804  165s *best
  ep4 step 200/628 loss 0.0520
  ep4 step 400/628 loss 0.0530
  ep4 step 600/628 loss 0.0537
ep  4/12  loss 0.0534  val macro-F1 0.8897  166s *best
  ep5 step 200/628 loss 0.0476
  ep5 step 400/628 loss 0.0455
  ep5 step 600/628 loss 0.0451
ep  5/12  loss 0.0452  val macro-F1 0.8758  164s
  ep6 step 200/628 loss 0.0357
  ep6 step 400/628 loss 0.0347
  ep6 step 600/628 loss 0.0354
ep  6/12  loss 0.0356  val macro-F1 0.9068  165s *best
  ep7 step 200/628 loss 0.0319
  ep7 step 400/628 lo

## 13. Evaluation

A multi-label head has no single 4x4 confusion matrix, because an image can be in two rows at
once. So we report two things instead: a 2x2 confusion matrix per class, which is the honest
per-label view, and one 5x5 matrix over the dominant class (largest defect area, with clean
as a fifth state), which is readable but only exact for the 93.6% of defect images that carry
a single class.

Both ROC and PR are reported. Under this much skew the PR curve is the more honest one: class
2 is 47 positives in 2,514 images, so a classifier that finds none of them still scores a
respectable ROC-AUC because the huge negative pool keeps the false-positive rate low. PR
compares against the 1.9% base rate instead and collapses when the positives are missed.

In [18]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, multilabel_confusion_matrix,
                             confusion_matrix)

def thin(x, n=200):
    '''Downsample a curve so the json stays readable without changing its shape.'''
    x = np.asarray(x, dtype=float)
    if len(x) <= n:
        return [round(float(v), 5) for v in x]
    k = np.linspace(0, len(x) - 1, n).astype(int)
    return [round(float(v), 5) for v in x[k]]


def evaluate(y, p, meta, thr=0.5):
    yhat = (p >= thr).astype(int)
    cls = [str(c) for c in range(1, 5)]
    m = dict(meta)
    m['n_val_images'] = int(len(y))
    m['threshold'] = thr
    m['accuracy'] = dict(
        subset_exact_match=float((yhat == y).all(1).mean()),
        per_class={c: float((yhat[:, i] == y[:, i]).mean()) for i, c in enumerate(cls)},
        macro=float(np.mean([(yhat[:, i] == y[:, i]).mean() for i in range(4)])))
    m['f1'] = dict(
        per_class={c: float(f1_score(y[:, i], yhat[:, i], zero_division=0))
                   for i, c in enumerate(cls)},
        macro=float(f1_score(y, yhat, average='macro', zero_division=0)),
        micro=float(f1_score(y, yhat, average='micro', zero_division=0)))
    m['precision'] = {c: float(precision_score(y[:, i], yhat[:, i], zero_division=0))
                      for i, c in enumerate(cls)}
    m['recall'] = {c: float(recall_score(y[:, i], yhat[:, i], zero_division=0))
                   for i, c in enumerate(cls)}
    m['support'] = {c: int(y[:, i].sum()) for i, c in enumerate(cls)}
    m['roc_auc'] = dict(
        per_class={c: float(roc_auc_score(y[:, i], p[:, i])) for i, c in enumerate(cls)},
        macro=float(np.mean([roc_auc_score(y[:, i], p[:, i]) for i in range(4)])))
    m['pr_auc_average_precision'] = dict(
        per_class={c: float(average_precision_score(y[:, i], p[:, i]))
                   for i, c in enumerate(cls)},
        macro=float(np.mean([average_precision_score(y[:, i], p[:, i]) for i in range(4)])))

    mcm = multilabel_confusion_matrix(y, yhat)
    m['confusion_matrix'] = dict(
        per_class_2x2={c: dict(tn=int(mcm[i][0, 0]), fp=int(mcm[i][0, 1]),
                               fn=int(mcm[i][1, 0]), tp=int(mcm[i][1, 1]))
                       for i, c in enumerate(cls)},
        note=('multi-label head: a 2x2 matrix per class is the exact view. The 5x5 matrix '
              'below is over the dominant class (largest defect area, 0 = clean) and is '
              'exact only for the 93.6% of defect images carrying a single class.'))
    pred_dom = np.where(p.max(1) >= thr, p.argmax(1) + 1, 0)
    m['confusion_matrix']['dominant_class_5x5'] = confusion_matrix(
        TRUE_DOM, pred_dom, labels=[0, 1, 2, 3, 4]).tolist()
    m['confusion_matrix']['dominant_class_5x5_labels'] = ['clean', 'c1', 'c2', 'c3', 'c4']

    curves = {}
    for i, c in enumerate(cls):
        fpr, tpr, _ = roc_curve(y[:, i], p[:, i])
        pr, rc, _ = precision_recall_curve(y[:, i], p[:, i])
        curves[c] = dict(roc=dict(fpr=thin(fpr), tpr=thin(tpr)),
                         pr=dict(precision=thin(pr), recall=thin(rc)))
    m['curves'] = curves

    any_p, any_y = p.max(1), y.max(1)
    m['derived_binary_has_defect'] = dict(
        accuracy=float(accuracy_score(any_y, any_p >= thr)),
        f1=float(f1_score(any_y, any_p >= thr, zero_division=0)),
        roc_auc=float(roc_auc_score(any_y, any_p)),
        pr_auc=float(average_precision_score(any_y, any_p)))
    return m


# phase-1 `primary_class` = the class holding the largest defect area, 0 for a clean image.
TRUE_DOM = idx.loc[val_ids, 'primary_class'].values.astype(int)

shared = dict(input_size=[IMG_H, IMG_W], grayscale_to_rgb='channel replication',
              split='splits/val.csv (frozen, SEED=42)', seed=SEED,
              augmentation=['HorizontalFlip', 'VerticalFlip', 'RandomBrightnessContrast'],
              optimizer='AdamW', scheduler='CosineAnnealingLR',
              head='4 sigmoids + BCE (multi-label)')

M = {}
M['resnet50'] = evaluate(y_val, prob_r50, dict(res_r50, model='ResNet-50 (ImageNet)', **shared))
M['effnetb2'] = evaluate(y_val, prob_b2, dict(res_b2, model='EfficientNet-B2 (ImageNet)', **shared))
M['resnet50']['binary_baseline'] = res_bin
M['resnet50']['imbalance_ablation'] = {
    k: dict(best_val_macro_f1=v['best_val_macro_f1'],
            per_class_f1=[float(x) for x in f1_score(
                y_val, (ABL_PROBS[k] >= 0.5).astype(int), average=None, zero_division=0)])
    for k, v in ABL.items()}
M['resnet50']['imbalance_ablation']['chosen'] = BEST_LOSS

for k, v in M.items():
    with open(f'{OUT}/metrics/cls_{k}.json', 'w') as f:
        json.dump(v, f, indent=2)
    print(f'wrote cls_{k}.json')

print(f'\n{"metric":28s} {"ResNet-50":>12s} {"EffNet-B2":>12s}')
def row(name, fn):
    print(f'{name:28s} {fn(M["resnet50"]):12.4f} {fn(M["effnetb2"]):12.4f}')
row('macro F1',            lambda m: m['f1']['macro'])
row('micro F1',            lambda m: m['f1']['micro'])
row('subset accuracy',     lambda m: m['accuracy']['subset_exact_match'])
row('macro per-class acc', lambda m: m['accuracy']['macro'])
row('macro ROC-AUC',       lambda m: m['roc_auc']['macro'])
row('macro PR-AUC',        lambda m: m['pr_auc_average_precision']['macro'])
for c in range(1, 5):
    row(f'  F1 class {c} (n={M["resnet50"]["support"][str(c)]})',
        lambda m, c=c: m['f1']['per_class'][str(c)])

wrote cls_resnet50.json
wrote cls_effnetb2.json

metric                          ResNet-50    EffNet-B2
macro F1                           0.9037       0.9341
micro F1                           0.9146       0.9440
subset accuracy                    0.9181       0.9443
macro per-class acc                0.9768       0.9844
macro ROC-AUC                      0.9906       0.9959
macro PR-AUC                       0.9522       0.9792
  F1 class 1 (n=179)               0.8786       0.8994
  F1 class 2 (n=47)                0.8738       0.9200
  F1 class 3 (n=1034)              0.9186       0.9495
  F1 class 4 (n=157)               0.9439       0.9675


## 14. Figures

In [19]:
NAMES = {'resnet50': 'ResNet-50', 'effnetb2': 'EfficientNet-B2'}
PROBS = {'resnet50': prob_r50, 'effnetb2': prob_b2}
COL = {'resnet50': '#1f77b4', 'effnetb2': '#d62728'}

# --- confusion matrices
for key, m in M.items():
    fig, ax = plt.subplots(1, 5, figsize=(19, 3.6))
    for i, c in enumerate(range(1, 5)):
        d = m['confusion_matrix']['per_class_2x2'][str(c)]
        cm = np.array([[d['tn'], d['fp']], [d['fn'], d['tp']]])
        ax[i].imshow(cm, cmap='Blues')
        for a in range(2):
            for b in range(2):
                ax[i].text(b, a, f'{cm[a, b]:,}', ha='center', va='center',
                           color='white' if cm[a, b] > cm.max() / 2 else 'black')
        ax[i].set(xticks=[0, 1], yticks=[0, 1], xticklabels=['pred 0', 'pred 1'],
                  yticklabels=['true 0', 'true 1'],
                  title=f'class {c}  (n={m["support"][str(c)]})')
    cm5 = np.array(m['confusion_matrix']['dominant_class_5x5'])
    lab = m['confusion_matrix']['dominant_class_5x5_labels']
    ax[4].imshow(np.log1p(cm5), cmap='Blues')
    for a in range(5):
        for b in range(5):
            ax[4].text(b, a, f'{cm5[a, b]}', ha='center', va='center', fontsize=8,
                       color='white' if np.log1p(cm5[a, b]) > np.log1p(cm5.max()) / 2
                       else 'black')
    ax[4].set(xticks=range(5), yticks=range(5), xticklabels=lab, yticklabels=lab,
              title='dominant class (log colour)')
    fig.suptitle(f'{NAMES[key]} — confusion matrices on the frozen val set')
    fig.tight_layout()
    fig.savefig(f'{OUT}/figs/cls_confusion_{key}.png', dpi=130, bbox_inches='tight')
    plt.close(fig)

# --- ROC and PR, both models overlaid per class
for kind in ('roc', 'pr'):
    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    for i, c in enumerate(range(1, 5)):
        for key, m in M.items():
            cv = m['curves'][str(c)]
            if kind == 'roc':
                auc = m['roc_auc']['per_class'][str(c)]
                ax[i].plot(cv['roc']['fpr'], cv['roc']['tpr'], color=COL[key],
                           label=f'{NAMES[key]} AUC={auc:.3f}')
            else:
                ap = m['pr_auc_average_precision']['per_class'][str(c)]
                ax[i].plot(cv['pr']['recall'], cv['pr']['precision'], color=COL[key],
                           label=f'{NAMES[key]} AP={ap:.3f}')
        n = M['resnet50']['support'][str(c)]
        base = n / M['resnet50']['n_val_images']
        if kind == 'roc':
            ax[i].plot([0, 1], [0, 1], 'k--', lw=0.8, label='chance')
            ax[i].set(xlabel='false positive rate', ylabel='true positive rate')
        else:
            ax[i].axhline(base, ls='--', color='k', lw=0.8,
                          label=f'base rate {base:.3f}')
            ax[i].set(xlabel='recall', ylabel='precision', ylim=(-0.02, 1.02))
        ax[i].set_title(f'class {c}  ({n} positives of 2514)')
        ax[i].legend(fontsize=8, loc='lower right' if kind == 'roc' else 'upper right')
    fig.suptitle(('ROC per defect class' if kind == 'roc' else
                  'Precision-recall per defect class (the honest view under this skew)'))
    fig.tight_layout()
    fig.savefig(f'{OUT}/figs/cls_{kind}_compare.png', dpi=130, bbox_inches='tight')
    plt.close(fig)

# --- training curves + ablation
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for key, r in (('resnet50', res_r50), ('effnetb2', res_b2)):
    h = pd.DataFrame(r['history'])
    ax[0].plot(h.epoch, h.train_loss, color=COL[key], label=NAMES[key])
    ax[1].plot(h.epoch, h.val_macro_f1, color=COL[key], marker='o', label=NAMES[key])
ax[0].set(xlabel='epoch', ylabel='train loss', title='training loss'); ax[0].legend()
ax[1].set(xlabel='epoch', ylabel='val macro F1', title='validation macro F1'); ax[1].legend()
ks = list(ABL)
ax[2].bar(ks, [ABL[k]['best_val_macro_f1'] for k in ks],
          color=['#999' if k != BEST_LOSS else '#1f77b4' for k in ks])
for i, k in enumerate(ks):
    ax[2].text(i, ABL[k]['best_val_macro_f1'], f'{ABL[k]["best_val_macro_f1"]:.3f}',
               ha='center', va='bottom')
ax[2].set(ylabel='val macro F1', title='imbalance treatment (ResNet-50, 5 epochs)')
fig.tight_layout()
fig.savefig(f'{OUT}/figs/cls_training_curves.png', dpi=130, bbox_inches='tight')
plt.close(fig)

print('figures written:')
for f in sorted(os.listdir(f'{OUT}/figs')):
    print(' ', f, '%.0f KB' % (os.path.getsize(f'{OUT}/figs/{f}') / 1024))

figures written:
  cls_confusion_effnetb2.png 54 KB
  cls_confusion_resnet50.png 55 KB
  cls_pr_compare.png 88 KB
  cls_roc_compare.png 93 KB
  cls_training_curves.png 98 KB


## 15. Export the artifacts

In [20]:
import shutil
BUNDLE = '/content/phase3_artifacts.zip'
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ('metrics', 'figs'):
        for f in sorted(os.listdir(f'{OUT}/{sub}')):
            z.write(f'{OUT}/{sub}/{f}', f'outputs/{sub}/{f}')
size = os.path.getsize(BUNDLE)
b64 = base64.b64encode(open(BUNDLE, 'rb').read()).decode()
print(f'bundle {size / 1024:.0f} KB -> base64 {len(b64) / 1024:.0f} KB')
print('BUNDLE_MD5', hashlib.md5(open(BUNDLE, "rb").read()).hexdigest())
print('BUNDLE_B64_START')
for i in range(0, len(b64), 4000):
    print(b64[i:i + 4000])
print('BUNDLE_B64_END')

bundle 514 KB -> base64 686 KB
BUNDLE_MD5 dbd47fad09f40a25d177483644ed791c
BUNDLE_B64_START
[base64 payload trimmed after unpacking into outputs/]
BUNDLE_B64_END
